In [0]:
%run "../00_Configuration/00_parametres"

In [0]:
from pyspark.sql.functions import col, upper, when, regexp_replace, trim

print("🧹 Démarrage du nettoyage (Couche SILVER)...\n")

# ==========================================
# 1. NETTOYAGE DES STATIONS (Filtre GPS + Normalisation)
# ==========================================
try:
    df_stations = spark.table(f"{DB_BRONZE}.raw_stations")
    
    # Règle 1 : Garder uniquement les points GPS qui sont au Maroc
    # Règle 2 : Normaliser le nom des enseignes concurrentes
    df_stations_clean = df_stations.filter(
        (col("lat").cast("float").between(21.0, 36.0)) & 
        (col("lon").cast("float").between(-17.0, -1.0))
    ).withColumn(
        "enseigne_norm", upper(col("enseigne"))
    ).withColumn(
        "enseigne_norm",
        when(col("enseigne_norm").contains("AFRIQUIA"), "AFRIQUIA")
        .when(col("enseigne_norm").contains("SHELL"), "SHELL")
        .when(col("enseigne_norm").contains("TOTAL"), "TOTALENERGIES")
        .when(col("enseigne_norm").contains("WINXO"), "WINXO")
        .when(col("enseigne_norm").contains("OLA"), "OLA ENERGY")
        .when(col("enseigne_norm").contains("ZIZ"), "ZIZ")
        .otherwise("INDEPENDANT")
    )
    
    df_stations_clean.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DB_SILVER}.slv_stations")
    print("✅ Nettoyage terminé : slv_stations")
except Exception as e:
    print(f"❌ Erreur sur les stations : {e}")

# ==========================================
# 2. NETTOYAGE DES COMMUNES (Typage et Calculs)
# ==========================================
try:
    df_communes = spark.table(f"{DB_BRONZE}.raw_communes")
    
    # Convertir en nombres et calculer la densité si elle n'existe pas
    df_communes_clean = df_communes.withColumn(
        "population", col("population").cast("int")
    ).withColumn(
        "superficie_km2", col("superficie_km2").cast("float")
    ).withColumn(
        "densite_hab_km2", col("population") / col("superficie_km2")
    ).filter(col("population").isNotNull())
    
    df_communes_clean.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DB_SILVER}.slv_communes")
    print("✅ Nettoyage terminé : slv_communes")
except Exception as e:
    print(f"❌ Erreur sur les communes : {e}")

# ==========================================
# 3. NETTOYAGE DU TRAFIC (TMJA) - CORRIGÉ
# ==========================================
try:
    df_trafic = spark.table(f"{DB_BRONZE}.raw_trafic_tmja")
    
    # On appelle le nom exact de la colonne avec le slash '/'
    df_trafic_clean = df_trafic.withColumn(
        "tmja_veh_j", 
        regexp_replace(col("tmja_en_véh/j"), " ", "").cast("int")
    ).filter(col("tmja_veh_j").isNotNull())
    
    df_trafic_clean.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DB_SILVER}.slv_trafic")
    print("✅ Nettoyage terminé : slv_trafic")
except Exception as e:
    print(f"❌ Erreur sur le trafic : {e}")

In [0]:
%run "../00_Configuration/00_parametres"

In [0]:
from pyspark.sql.functions import col, upper, when, regexp_replace, round

print("🧹 Nettoyage Silver en mode 'Force' (Bronze -> Silver)...\n")

# --- 1. NETTOYAGE DES STATIONS (Correction du Schéma) ---
try:
    df_stations = spark.table(f"{DB_BRONZE}.raw_stations")
    
    df_stations_slv = df_stations.filter(
        (col("lat").cast("float").between(21.0, 36.0)) & 
        (col("lon").cast("float").between(-17.0, -1.0))
    ).withColumn("enseigne_propre", 
        when(upper(col("enseigne")).contains("AFRIQUIA"), "AFRIQUIA")
        .when(upper(col("enseigne")).contains("SHELL"), "SHELL")
        .when(upper(col("enseigne")).contains("TOTAL"), "TOTALENERGIES")
        .when(upper(col("enseigne")).contains("WINXO"), "WINXO")
        .otherwise("AUTRE/INDEPENDANT")
    )
    
    # L'option overwriteSchema permet de corriger l'erreur de mismatch
    df_stations_slv.write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true").saveAsTable(f"{DB_SILVER}.slv_stations")
    print("✅ Succès : slv_stations (Schéma mis à jour)")
except Exception as e: print(f"❌ Erreur Stations : {e}")

# --- 2. NETTOYAGE DES COMMUNES (Correction du Type) ---
try:
    df_communes = spark.table(f"{DB_BRONZE}.raw_communes")
    
    df_communes_slv = df_communes.withColumn("pop_int", col("population").cast("int")) \
        .withColumn("surf_float", col("superficie_km2").cast("float")) \
        .withColumn("densite", round(col("pop_int") / col("surf_float"), 2))
    
    df_communes_slv.write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true").saveAsTable(f"{DB_SILVER}.slv_communes")
    print("✅ Succès : slv_communes")
except Exception as e: print(f"❌ Erreur Communes : {e}")

# --- 3. NETTOYAGE DU TRAFIC (Correction du nom de colonne) ---
try:
    df_trafic = spark.table(f"{DB_BRONZE}.raw_trafic_tmja")
    
    # On utilise le nom de colonne suggéré par l'erreur : `tmja_en_véh/j`
    df_trafic_slv = df_trafic.withColumn("tmja_final", 
        regexp_replace(col("`tmja_en_véh/j`"), " ", "").cast("int")
    ).filter(col("tmja_final").isNotNull())
    
    df_trafic_slv.write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true").saveAsTable(f"{DB_SILVER}.slv_trafic")
    print("✅ Succès : slv_trafic")
except Exception as e: print(f"❌ Erreur Trafic : {e}")

print("\n🚀 Couche SILVER validée et prête !")